# Part 4.1 - Spatial Modelling

Catalyst v16 uses `DiscreteSpaceReactionSystem` for lattice-based spatial CRN models.

We combine:
- a non-spatial reaction network,
- transport reactions,
- and a lattice.


In [ ]:
using Catalyst, OrdinaryDiffEq, JumpProcesses, Plots

brusselator = @reaction_network begin
    A, 0 --> X
    1, 2X + Y --> 3X
    B, X --> Y
    1, X --> 0
end

tr = @transport_reaction D X
lattice = CartesianGrid((20, 20))
dsrs = DiscreteSpaceReactionSystem(brusselator, [tr], lattice)


## Deterministic spatial simulation (ODE)

Here `X` starts with heterogeneous values across the grid, while `Y` starts uniformly.


In [ ]:
u0 = [:X => 10 .* rand(20, 20), :Y => 5.0]
ps = [:A => 1.0, :B => 4.0, :D => 0.2]

ode_prob = ODEProblem(dsrs, u0, (0.0, 50.0), ps)
ode_sol = solve(ode_prob, QNDF(); saveat = 0.5)

plot(ode_sol; idxs = dsrs.X[10, 10], xlabel = "t", ylabel = "X", title = "X at lattice site (10,10)")


In [ ]:
heatmap(
    ode_sol[:X][end];
    xlabel = "i",
    ylabel = "j",
    title = "Final X profile (ODE)",
    colorbar_title = "X",
)


## Spatial SSA simulation

For jump simulations, we create a spatial jump problem with an SSA solver (here `NSM()`).


In [ ]:
u0_jump = [:X => rand(1:10, 20, 20), :Y => 5]
ps_jump = [:A => 0.1, :B => 2.0, :D => 0.05]

jump_prob = JumpProblem(
    dsrs,
    u0_jump,
    (0.0, 30.0),
    ps_jump,
    NSM();
    save_positions = (false, false),
)

jump_sol = solve(jump_prob, SSAStepper(); saveat = 0.2)

plot(jump_sol; idxs = dsrs.X[10, 10], xlabel = "t", ylabel = "X", title = "X at lattice site (10,10), jump")


In [ ]:
heatmap(
    jump_sol[:X][end];
    xlabel = "i",
    ylabel = "j",
    title = "Final X profile (jump)",
    colorbar_title = "X",
)
